In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from timeseries import read_timeseries_csv
from scenarios.price_scenarios import build_zonal_price_matrix, zone_annual_cf, CONTRACT_START

own_zone = "nord"
reference_zone = "sicilia"

price_own_zone = build_zonal_price_matrix(own_zone)
price_reference_zone = build_zonal_price_matrix(reference_zone)
price_own_zone.head()

price_scenario Delayed transition                                      \
weather_year                 1991        1992        1993        1994   
contract_year                                                           
2026                   103.875909  104.316244  104.055878  104.264543   
2027                   101.454880  101.895215  101.634849  101.843513   
2028                    99.033851   99.474185   99.213819   99.422484   
2029                    96.612821   97.053156   96.792790   97.001455   
2030                    94.191792   94.632127   94.371761   94.580426   

price_scenario                                                              \
weather_year          1995        1996        1997        1998        1999   
contract_year                                                                
2026            103.783605  104.723265  103.665408  103.587186  104.067303   
2027            101.362575  102.302236  101.244378  101.166157  101.646274   
2028             98.941546   99.881206   98.823349   98.745128   99.225244   
2029             96.520517   97.460177   96.402320   96.324099   96.804215   
2030             94.099488   95.039148   93.981291   93.903069   94.383186   

price_scenario              ... Net Zero 2050                          \
weather_year          2000  ...          2014        2015        2016   
contract_year               ...                                         
2026            104.040144  ...    110.643072  109.916631  110.109090   
2027            101.619114  ...    111.375168  110.648728  110.841187   
2028             99.198085  ...    112.107264  111.380824  111.573283   
2029             96.777056  ...    112.839361  112.112920  112.305379   
2030             94.356027  ...    113.571457  112.845017  113.037476   

price_scenario                                                              \
weather_year          2017        2018        2019        2020        2021   
contract_year                                                                
2026            109.508766  110.246683  109.840647  109.721979  109.802884   
2027            110.240862  110.978779  110.572743  110.454075  110.534981   
2028            110.972959  111.710875  111.304839  111.186172  111.267077   
2029            111.705055  112.442972  112.036936  111.918268  111.999173   
2030            112.437151  113.175068  112.769032  112.650364  112.731270   

price_scenario                          
weather_year          2022        2023  
contract_year                           
2026            109.546311  109.692967  
2027            110.278407  110.425063  
2028            111.010504  111.157160  
2029            111.742600  111.889256  
2030            112.474696  112.621352  

[5 rows x 99 columns]

In [2]:
annual_solar_cf = zone_annual_cf("solar", own_zone)
annual_solar_cf.describe()

count    33.000000
mean      0.181007
std       0.006687
min       0.167394
25%       0.176553
50%       0.181215
75%       0.186130
max       0.190684
dtype: float64

In [3]:
STRIKE_PRICE_SOLAR = 56.83
COUNTERPARTY_SPREAD = 5.0
HOURS_PER_YEAR = 8760

pap_payment_per_mw = (STRIKE_PRICE_SOLAR + COUNTERPARTY_SPREAD) * annual_solar_cf * HOURS_PER_YEAR
pap_payment_per_mw.describe()

count        33.000000
mean      98039.063751
std        3622.019858
min       90665.745591
25%       95626.435671
50%       98151.705732
75%      100813.548487
max      103280.386110
dtype: float64

In [4]:
from scenarios.load_profile import load_profile_for_archetype

annual_kwh = 20_000_000
load = load_profile_for_archetype("chemicals", annual_kwh)
load.sum() / 1000

np.float64(20000.0)

In [5]:
def annual_residual_mwh(technology, zone, contracted_mw):
    load_mwh = load.to_numpy() / 1000
    by_year = {}
    for year in range(1991, 2024):
        cf = read_timeseries_csv(PROJECT_ROOT / "data" / "processed" / technology / zone / f"{technology}_cf_{year}.csv")
        gen_mwh = cf[f"{technology}_cf"].resample("h").mean().to_numpy() * contracted_mw
        n = min(len(load_mwh), len(gen_mwh))
        by_year[year] = np.maximum(load_mwh[:n] - gen_mwh[:n], 0.0).sum()
    return pd.Series(by_year)

In [6]:
annual_load_mwh = load.sum() / 1000
CONTRACTED_MW = 2

residual_mwh = annual_residual_mwh("solar", own_zone, CONTRACTED_MW)
residual_mwh.describe()

count       33.000000
mean     16841.011818
std        116.725494
min      16676.522163
25%      16748.298320
50%      16840.520300
75%      16920.897324
max      17074.060399
dtype: float64

In [7]:
payment_total = pap_payment_per_mw * CONTRACTED_MW

columns = {}
for scenario, weather_year in price_own_zone.columns:
    spot_price = price_own_zone[(scenario, weather_year)]
    columns[(scenario, weather_year)] = payment_total[weather_year] + residual_mwh[weather_year] * spot_price

pap_solar_cost_matrix = pd.DataFrame(columns)
pap_solar_cost_matrix.columns.names = ["price_scenario", "weather_year"]
pap_solar_cost_matrix.index.name = "contract_year"
pap_solar_cost_matrix.head()

price_scenario Delayed transition                                            \
weather_year                 1991          1992          1993          1994   
contract_year                                                                 
2026                 1.943208e+06  1.957084e+06  1.949030e+06  1.955502e+06   
2027                 1.902572e+06  1.916078e+06  1.908235e+06  1.914537e+06   
2028                 1.861937e+06  1.875073e+06  1.867440e+06  1.873571e+06   
2029                 1.821301e+06  1.834067e+06  1.826646e+06  1.832605e+06   
2030                 1.780665e+06  1.793061e+06  1.785851e+06  1.791639e+06   

price_scenario                                                          \
weather_year            1995          1996          1997          1998   
contract_year                                                            
2026            1.939826e+06  1.969383e+06  1.936245e+06  1.933428e+06   
2027            1.899278e+06  1.928046e+06  1.895790e+06  1.893050e+06   
2028            1.858730e+06  1.886709e+06  1.855335e+06  1.852673e+06   
2029            1.818182e+06  1.845372e+06  1.814881e+06  1.812295e+06   
2030            1.777633e+06  1.804036e+06  1.774426e+06  1.771917e+06   

price_scenario                              ... Net Zero 2050                \
weather_year            1999          2000  ...          2014          2015   
contract_year                               ...                               
2026            1.949174e+06  1.948345e+06  ...  2.069896e+06  2.045852e+06   
2027            1.908374e+06  1.907573e+06  ...  2.082385e+06  2.058161e+06   
2028            1.867574e+06  1.866801e+06  ...  2.094875e+06  2.070469e+06   
2029            1.826774e+06  1.826029e+06  ...  2.107364e+06  2.082778e+06   
2030            1.785974e+06  1.785257e+06  ...  2.119853e+06  2.095086e+06   

price_scenario                                                          \
weather_year            2016          2017          2018          2019   
contract_year                                                            
2026            2.052731e+06  2.032786e+06  2.057118e+06  2.043716e+06   
2027            2.065091e+06  2.044995e+06  2.069512e+06  2.056008e+06   
2028            2.077451e+06  2.057204e+06  2.081906e+06  2.068301e+06   
2029            2.089812e+06  2.069413e+06  2.094299e+06  2.080594e+06   
2030            2.102172e+06  2.081621e+06  2.106693e+06  2.092887e+06   

price_scenario                                                          
weather_year            2020          2021          2022          2023  
contract_year                                                           
2026            2.038791e+06  2.042292e+06  2.033081e+06  2.037877e+06  
2027            2.051046e+06  2.054572e+06  2.045291e+06  2.050124e+06  
2028            2.063301e+06  2.066853e+06  2.057501e+06  2.062372e+06  
2029            2.075556e+06  2.079134e+06  2.069712e+06  2.074620e+06  
2030            2.087812e+06  2.091415e+06  2.081922e+06  2.086868e+06  

[5 rows x 99 columns]

In [8]:
STRIKE_PRICE_WIND = 72.85

annual_wind_cf = zone_annual_cf("wind", own_zone)
wind_payment_per_mw = (STRIKE_PRICE_WIND + COUNTERPARTY_SPREAD) * annual_wind_cf * HOURS_PER_YEAR
residual_wind_mwh = annual_residual_mwh("wind", own_zone, CONTRACTED_MW)
residual_wind_mwh.describe()

count       33.000000
mean     18744.878683
std        118.261614
min      18533.963367
25%      18687.530829
50%      18748.902037
75%      18814.369350
max      18975.716635
dtype: float64

In [9]:
wind_payment_total = wind_payment_per_mw * CONTRACTED_MW

wind_columns = {}
for scenario, weather_year in price_own_zone.columns:
    spot_price = price_own_zone[(scenario, weather_year)]
    wind_columns[(scenario, weather_year)] = wind_payment_total[weather_year] + residual_wind_mwh[weather_year] * spot_price

pap_wind_cost_matrix = pd.DataFrame(wind_columns)
pap_wind_cost_matrix.columns.names = ["price_scenario", "weather_year"]
pap_wind_cost_matrix.index.name = "contract_year"
pap_wind_cost_matrix.head()

price_scenario Delayed transition                                            \
weather_year                 1991          1992          1993          1994   
contract_year                                                                 
2026                 2.047351e+06  2.057795e+06  2.045190e+06  2.052629e+06   
2027                 2.001749e+06  2.012114e+06  2.000115e+06  2.007311e+06   
2028                 1.956147e+06  1.966432e+06  1.955040e+06  1.961992e+06   
2029                 1.910545e+06  1.920751e+06  1.909965e+06  1.916674e+06   
2030                 1.864943e+06  1.875070e+06  1.864890e+06  1.871356e+06   

price_scenario                                                          \
weather_year            1995          1996          1997          1998   
contract_year                                                            
2026            2.045290e+06  2.058065e+06  2.036825e+06  2.038830e+06   
2027            1.999857e+06  2.012959e+06  1.991917e+06  1.993553e+06   
2028            1.954423e+06  1.967853e+06  1.947009e+06  1.948276e+06   
2029            1.908989e+06  1.922747e+06  1.902101e+06  1.902999e+06   
2030            1.863556e+06  1.877641e+06  1.857192e+06  1.857722e+06   

price_scenario                              ... Net Zero 2050                \
weather_year            1999          2000  ...          2014          2015   
contract_year                               ...                               
2026            2.044871e+06  2.054188e+06  ...  2.180053e+06  2.158526e+06   
2027            1.999904e+06  2.008334e+06  ...  2.193945e+06  2.172252e+06   
2028            1.954936e+06  1.962479e+06  ...  2.207837e+06  2.185978e+06   
2029            1.909968e+06  1.916625e+06  ...  2.221729e+06  2.199704e+06   
2030            1.865000e+06  1.870770e+06  ...  2.235621e+06  2.213430e+06   

price_scenario                                                          \
weather_year            2016          2017          2018          2019   
contract_year                                                            
2026            2.159767e+06  2.147192e+06  2.163987e+06  2.151570e+06   
2027            2.173449e+06  2.160834e+06  2.177702e+06  2.165153e+06   
2028            2.187130e+06  2.174476e+06  2.191416e+06  2.178736e+06   
2029            2.200811e+06  2.188118e+06  2.205130e+06  2.192318e+06   
2030            2.214492e+06  2.201760e+06  2.218844e+06  2.205901e+06   

price_scenario                                                          
weather_year            2020          2021          2022          2023  
contract_year                                                           
2026            2.154706e+06  2.158249e+06  2.158199e+06  2.155951e+06  
2027            2.168437e+06  2.172023e+06  2.172082e+06  2.169719e+06  
2028            2.182168e+06  2.185797e+06  2.185966e+06  2.183488e+06  
2029            2.195900e+06  2.199570e+06  2.199850e+06  2.197256e+06  
2030            2.209631e+06  2.213344e+06  2.213733e+06  2.211024e+06  

[5 rows x 99 columns]

In [10]:
BASELOAD_DISCOUNT = 0.14
CONTRACTED_MW_BASELOAD = 2

reference_baseload_price = price_own_zone.mean().mean()
strike_baseload = reference_baseload_price * (1 - BASELOAD_DISCOUNT) + COUNTERPARTY_SPREAD
strike_baseload

np.float64(87.94876862842975)

In [11]:
baseload_payment = strike_baseload * CONTRACTED_MW_BASELOAD * HOURS_PER_YEAR
residual_baseload_mwh = annual_load_mwh - CONTRACTED_MW_BASELOAD * HOURS_PER_YEAR

baseload_columns = {}
for scenario, weather_year in price_own_zone.columns:
    spot_price = price_own_zone[(scenario, weather_year)]
    baseload_columns[(scenario, weather_year)] = baseload_payment + residual_baseload_mwh * spot_price

baseload_cost_matrix = pd.DataFrame(baseload_columns)
baseload_cost_matrix.columns.names = ["price_scenario", "weather_year"]
baseload_cost_matrix.index.name = "contract_year"
baseload_cost_matrix.head()

price_scenario Delayed transition                                            \
weather_year                 1991          1992          1993          1994   
contract_year                                                                 
2026                 1.798475e+06  1.799567e+06  1.798921e+06  1.799438e+06   
2027                 1.792471e+06  1.793563e+06  1.792917e+06  1.793434e+06   
2028                 1.786466e+06  1.787558e+06  1.786913e+06  1.787430e+06   
2029                 1.780462e+06  1.781554e+06  1.780909e+06  1.781426e+06   
2030                 1.774458e+06  1.775550e+06  1.774904e+06  1.775422e+06   

price_scenario                                                          \
weather_year            1995          1996          1997          1998   
contract_year                                                            
2026            1.798246e+06  1.800576e+06  1.797953e+06  1.797759e+06   
2027            1.792242e+06  1.794572e+06  1.791948e+06  1.791754e+06   
2028            1.786237e+06  1.788568e+06  1.785944e+06  1.785750e+06   
2029            1.780233e+06  1.782564e+06  1.779940e+06  1.779746e+06   
2030            1.774229e+06  1.776560e+06  1.773936e+06  1.773742e+06   

price_scenario                              ... Net Zero 2050                \
weather_year            1999          2000  ...          2014          2015   
contract_year                               ...                               
2026            1.798949e+06  1.798882e+06  ...  1.815257e+06  1.813456e+06   
2027            1.792945e+06  1.792878e+06  ...  1.817073e+06  1.815271e+06   
2028            1.786941e+06  1.786874e+06  ...  1.818888e+06  1.817087e+06   
2029            1.780937e+06  1.780870e+06  ...  1.820704e+06  1.818902e+06   
2030            1.774933e+06  1.774865e+06  ...  1.822520e+06  1.820718e+06   

price_scenario                                                          \
weather_year            2016          2017          2018          2019   
contract_year                                                            
2026            1.813933e+06  1.812444e+06  1.814274e+06  1.813267e+06   
2027            1.815749e+06  1.814260e+06  1.816090e+06  1.815083e+06   
2028            1.817564e+06  1.816075e+06  1.817905e+06  1.816898e+06   
2029            1.819380e+06  1.817891e+06  1.819721e+06  1.818714e+06   
2030            1.821195e+06  1.819707e+06  1.821537e+06  1.820530e+06   

price_scenario                                                          
weather_year            2020          2021          2022          2023  
contract_year                                                           
2026            1.812973e+06  1.813174e+06  1.812537e+06  1.812901e+06  
2027            1.814789e+06  1.814989e+06  1.814353e+06  1.814717e+06  
2028            1.816604e+06  1.816805e+06  1.816168e+06  1.816532e+06  
2029            1.818420e+06  1.818620e+06  1.817984e+06  1.818348e+06  
2030            1.820235e+06  1.820436e+06  1.819800e+06  1.820163e+06  

[5 rows x 99 columns]

In [12]:
SLEEVING_MARGIN = 3.0

sleeved_solar_mwh = annual_solar_cf * HOURS_PER_YEAR * CONTRACTED_MW
sleeving_fee_total = SLEEVING_MARGIN * sleeved_solar_mwh
sleeving_fee_total.describe()

count       33.000000
mean      9513.737385
std        351.481791
min       8798.228587
25%       9279.615300
50%       9524.668193
75%       9782.974137
max      10022.356731
dtype: float64

In [13]:
sleeved_columns = {}
for scenario, weather_year in price_own_zone.columns:
    sleeved_columns[(scenario, weather_year)] = (
        pap_solar_cost_matrix[(scenario, weather_year)] + sleeving_fee_total[weather_year]
    )

sleeved_cost_matrix = pd.DataFrame(sleeved_columns)
sleeved_cost_matrix.columns.names = ["price_scenario", "weather_year"]
sleeved_cost_matrix.index.name = "contract_year"
sleeved_cost_matrix.head()

price_scenario Delayed transition                                            \
weather_year                 1991          1992          1993          1994   
contract_year                                                                 
2026                 1.952898e+06  1.966315e+06  1.958524e+06  1.964782e+06   
2027                 1.912262e+06  1.925309e+06  1.917729e+06  1.923816e+06   
2028                 1.871626e+06  1.884303e+06  1.876935e+06  1.882850e+06   
2029                 1.830991e+06  1.843297e+06  1.836140e+06  1.841884e+06   
2030                 1.790355e+06  1.802292e+06  1.795346e+06  1.800918e+06   

price_scenario                                                          \
weather_year            1995          1996          1997          1998   
contract_year                                                            
2026            1.949609e+06  1.978181e+06  1.946144e+06  1.943414e+06   
2027            1.909061e+06  1.936844e+06  1.905689e+06  1.903036e+06   
2028            1.868513e+06  1.895507e+06  1.865235e+06  1.862659e+06   
2029            1.827964e+06  1.854171e+06  1.824780e+06  1.822281e+06   
2030            1.787416e+06  1.812834e+06  1.784325e+06  1.781904e+06   

price_scenario                              ... Net Zero 2050                \
weather_year            1999          2000  ...          2014          2015   
contract_year                               ...                               
2026            1.958655e+06  1.957866e+06  ...  2.078746e+06  2.055453e+06   
2027            1.917855e+06  1.917094e+06  ...  2.091236e+06  2.067761e+06   
2028            1.877055e+06  1.876322e+06  ...  2.103725e+06  2.080070e+06   
2029            1.836255e+06  1.835551e+06  ...  2.116214e+06  2.092378e+06   
2030            1.795455e+06  1.794779e+06  ...  2.128703e+06  2.104687e+06   

price_scenario                                                          \
weather_year            2016          2017          2018          2019   
contract_year                                                            
2026            2.062128e+06  2.042808e+06  2.066374e+06  2.053389e+06   
2027            2.074489e+06  2.055017e+06  2.078768e+06  2.065681e+06   
2028            2.086849e+06  2.067226e+06  2.091161e+06  2.077974e+06   
2029            2.099210e+06  2.079435e+06  2.103555e+06  2.090267e+06   
2030            2.111570e+06  2.091644e+06  2.115948e+06  2.102560e+06   

price_scenario                                                          
weather_year            2020          2021          2022          2023  
contract_year                                                           
2026            2.048595e+06  2.052013e+06  2.043075e+06  2.047713e+06  
2027            2.060850e+06  2.064294e+06  2.055286e+06  2.059961e+06  
2028            2.073105e+06  2.076575e+06  2.067496e+06  2.072208e+06  
2029            2.085361e+06  2.088856e+06  2.079706e+06  2.084456e+06  
2030            2.097616e+06  2.101137e+06  2.091917e+06  2.096704e+06  

[5 rows x 99 columns]

In [14]:
basis_risk = price_own_zone - price_reference_zone
basis_risk.loc[CONTRACT_START].describe()

count    9.900000e+01
mean     2.153160e-15
std      4.270334e-01
min     -6.005946e-01
25%     -4.115476e-01
50%     -1.651287e-02
75%      1.980173e-01
max      1.023159e+00
Name: 2026, dtype: float64

In [15]:
CONTRACTED_MW_VPPA = 5

wind_cf_reference = zone_annual_cf("wind", reference_zone)

vppa_columns = {}
for scenario, weather_year in price_own_zone.columns:
    spot_own = price_own_zone[(scenario, weather_year)]
    spot_reference = price_reference_zone[(scenario, weather_year)]
    settlement = (STRIKE_PRICE_WIND + COUNTERPARTY_SPREAD - spot_reference) * wind_cf_reference[weather_year] * HOURS_PER_YEAR * CONTRACTED_MW_VPPA
    vppa_columns[(scenario, weather_year)] = annual_load_mwh * spot_own + settlement

vppa_cost_matrix = pd.DataFrame(vppa_columns)
vppa_cost_matrix.columns.names = ["price_scenario", "weather_year"]
vppa_cost_matrix.index.name = "contract_year"
vppa_cost_matrix.head()

price_scenario Delayed transition                                            \
weather_year                 1991          1992          1993          1994   
contract_year                                                                 
2026                 1.816634e+06  1.824106e+06  1.813103e+06  1.819645e+06   
2027                 1.792219e+06  1.799801e+06  1.789428e+06  1.795740e+06   
2028                 1.767804e+06  1.775497e+06  1.765752e+06  1.771835e+06   
2029                 1.743388e+06  1.751192e+06  1.742076e+06  1.747930e+06   
2030                 1.718973e+06  1.726887e+06  1.718401e+06  1.724025e+06   

price_scenario                                                          \
weather_year            1995          1996          1997          1998   
contract_year                                                            
2026            1.789654e+06  1.753578e+06  1.809725e+06  1.797313e+06   
2027            1.767747e+06  1.737083e+06  1.785575e+06  1.774286e+06   
2028            1.745841e+06  1.720589e+06  1.761425e+06  1.751260e+06   
2029            1.723934e+06  1.704095e+06  1.737275e+06  1.728233e+06   
2030            1.702027e+06  1.687601e+06  1.713125e+06  1.705206e+06   

price_scenario                              ... Net Zero 2050                \
weather_year            1999          2000  ...          2014          2015   
contract_year                               ...                               
2026            1.804180e+06  1.805282e+06  ...  1.862036e+06  1.878542e+06   
2027            1.781438e+06  1.782364e+06  ...  1.868672e+06  1.885928e+06   
2028            1.758696e+06  1.759446e+06  ...  1.875307e+06  1.893314e+06   
2029            1.735955e+06  1.736528e+06  ...  1.881942e+06  1.900699e+06   
2030            1.713213e+06  1.713610e+06  ...  1.888578e+06  1.908085e+06   

price_scenario                                                          \
weather_year            2016          2017          2018          2019   
contract_year                                                            
2026            1.842541e+06  1.865588e+06  1.858031e+06  1.826318e+06   
2027            1.848971e+06  1.872841e+06  1.864775e+06  1.832479e+06   
2028            1.855400e+06  1.880093e+06  1.871519e+06  1.838641e+06   
2029            1.861830e+06  1.887345e+06  1.878264e+06  1.844802e+06   
2030            1.868260e+06  1.894598e+06  1.885008e+06  1.850963e+06   

price_scenario                                                          
weather_year            2020          2021          2022          2023  
contract_year                                                           
2026            1.903912e+06  1.837400e+06  1.889691e+06  1.895687e+06  
2027            1.911982e+06  1.843856e+06  1.897505e+06  1.903577e+06  
2028            1.920053e+06  1.850312e+06  1.905318e+06  1.911466e+06  
2029            1.928123e+06  1.856768e+06  1.913132e+06  1.919356e+06  
2030            1.936193e+06  1.863224e+06  1.920946e+06  1.927246e+06  

[5 rows x 99 columns]

In [16]:
spot_only_columns = {}
for scenario, weather_year in price_own_zone.columns:
    spot_only_columns[(scenario, weather_year)] = annual_load_mwh * price_own_zone[(scenario, weather_year)]

spot_only_cost_matrix = pd.DataFrame(spot_only_columns)
spot_only_cost_matrix.columns.names = ["price_scenario", "weather_year"]
spot_only_cost_matrix.index.name = "contract_year"
spot_only_cost_matrix.head()

price_scenario Delayed transition                                            \
weather_year                 1991          1992          1993          1994   
contract_year                                                                 
2026                 2.077518e+06  2.086325e+06  2.081118e+06  2.085291e+06   
2027                 2.029098e+06  2.037904e+06  2.032697e+06  2.036870e+06   
2028                 1.980677e+06  1.989484e+06  1.984276e+06  1.988450e+06   
2029                 1.932256e+06  1.941063e+06  1.935856e+06  1.940029e+06   
2030                 1.883836e+06  1.892643e+06  1.887435e+06  1.891609e+06   

price_scenario                                                          \
weather_year            1995          1996          1997          1998   
contract_year                                                            
2026            2.075672e+06  2.094465e+06  2.073308e+06  2.071744e+06   
2027            2.027252e+06  2.046045e+06  2.024888e+06  2.023323e+06   
2028            1.978831e+06  1.997624e+06  1.976467e+06  1.974903e+06   
2029            1.930410e+06  1.949204e+06  1.928046e+06  1.926482e+06   
2030            1.881990e+06  1.900783e+06  1.879626e+06  1.878061e+06   

price_scenario                              ... Net Zero 2050                \
weather_year            1999          2000  ...          2014          2015   
contract_year                               ...                               
2026            2.081346e+06  2.080803e+06  ...  2.212861e+06  2.198333e+06   
2027            2.032925e+06  2.032382e+06  ...  2.227503e+06  2.212975e+06   
2028            1.984505e+06  1.983962e+06  ...  2.242145e+06  2.227616e+06   
2029            1.936084e+06  1.935541e+06  ...  2.256787e+06  2.242258e+06   
2030            1.887664e+06  1.887121e+06  ...  2.271429e+06  2.256900e+06   

price_scenario                                                          \
weather_year            2016          2017          2018          2019   
contract_year                                                            
2026            2.202182e+06  2.190175e+06  2.204934e+06  2.196813e+06   
2027            2.216824e+06  2.204817e+06  2.219576e+06  2.211455e+06   
2028            2.231466e+06  2.219459e+06  2.234218e+06  2.226097e+06   
2029            2.246108e+06  2.234101e+06  2.248859e+06  2.240739e+06   
2030            2.260750e+06  2.248743e+06  2.263501e+06  2.255381e+06   

price_scenario                                                          
weather_year            2020          2021          2022          2023  
contract_year                                                           
2026            2.194440e+06  2.196058e+06  2.190926e+06  2.193859e+06  
2027            2.209082e+06  2.210700e+06  2.205568e+06  2.208501e+06  
2028            2.223723e+06  2.225342e+06  2.220210e+06  2.223143e+06  
2029            2.238365e+06  2.239983e+06  2.234852e+06  2.237785e+06  
2030            2.253007e+06  2.254625e+06  2.249494e+06  2.252427e+06  

[5 rows x 99 columns]